# MTAH 2027 — train PossessionNet on Colab

Trains on **external data only** (IDSSE, other games). The competition data never comes here:
inference on BAR-ATH runs locally on CPU (`python -m mtah.infer`).

Before the first run:
1. Runtime → Change runtime type → **A100 GPU**.
2. Colab sidebar 🔑 *Secrets*: add `GH_TOKEN` = a GitHub fine-grained token with read access to the repo.
3. Upload the converted external matches (`cache/idsse_*.npz`, made locally with
   `python -m mtah.external.idsse`) to Drive at `MyDrive/Metrica-Hackathon/mtah/cache/` (see `DRIVE_ROOT`).

Training writes to `<DRIVE_ROOT>/outputs/<RUN>` and **resumes automatically** — after a disconnect just run all cells again.

In [ ]:
REPO = "hnfadtya/more-than-a-hack-2027"   # change if you move to a new private repo
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/Metrica-Hackathon/mtah"   # cache/ and outputs/ live here
RUN = "run1"
VAL = "idsse_J03WMX idsse_J03WR9"         # held-out matches (one Bundesliga, one 2. Bundesliga)
EXTRA_ARGS = "--epochs 30 --steps 500 --batch 32"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
import os, subprocess
token = userdata.get('GH_TOKEN')
if not os.path.isdir('/content/mtah'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', '/content/mtah'], check=True)
else:
    subprocess.run(['git', '-C', '/content/mtah', 'pull', '-q'], check=True)
%cd /content/mtah
!git log --oneline -1

In [ ]:
# Colab already ships torch, numpy, pandas, numba. Copy the external data to local disk for speed.
!mkdir -p /content/mtah/cache && cp -n {DRIVE_ROOT}/cache/idsse_*.npz /content/mtah/cache/ 2>/dev/null; ls -la /content/mtah/cache
!python -c "import torch, numpy, numba; print(torch.__version__, numpy.__version__, numba.__version__, torch.cuda.is_available())"

Quick smoke test on synthetic data (~1 min). Skip once it has passed.

In [ ]:
!python -m mtah.train --synthetic 3 --epochs 1 --steps 30 --batch 8 --out /content/smoke --fresh

Real training. Re-run this cell after a disconnect: it resumes from `last.pt`.

In [ ]:
OUT = f"{DRIVE_ROOT}/outputs/{RUN}"
!mkdir -p {OUT}
!python -m mtah.train --data "cache/idsse_*.npz" --val {VAL} --out {OUT} {EXTRA_ARGS} 2>&1 | tee -a {OUT}.log

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open(f"{OUT}/history.json"))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r['loss'] for r in h]); ax[0].set_title('train loss')
ax[1].plot([r['val_spellF1'] for r in h], label='spellF1'); ax[1].plot([r['val_macroF1'] for r in h], label='macroF1')
ax[1].legend(); ax[1].set_title('validation (decoded, official metric)')
plt.show()

Download `best.pt` from Drive to your laptop (`outputs/<RUN>/best.pt`) and run inference locally.